# Unity Catalog Governance Validation

## Purpose:
   Verify governance controls around the existing MDM data model.

 This notebook does NOT transform business data.\
 It only validates governance configuration.

In [0]:
from pyspark.sql import functions as F

CATALOG = "bfsi_mdm_dev"

print("Unity Catalog governance validation starting...")

Unity Catalog governance validation starting...


##Validate catalog existence

In [0]:
# ------------------------------------------------------------
# Verify that the project Unity Catalog catalog exists.
# ------------------------------------------------------------

catalog_exists = (
    spark.sql("SHOW CATALOGS")
    .filter(F.col("catalog") == CATALOG)
    .count()
    > 0
)

assert catalog_exists, (
    f"Catalog {CATALOG} does not exist."
)

print("Catalog existence: PASS")

Catalog existence: PASS


##Validate schemas

In [0]:
required_schemas = {
    "bronze",
    "silver",
    "mdm",
    "gold",
    "quarantine",
    "audit",
}

actual_schemas = {
    row["databaseName"]
    for row in spark.sql(
        f"SHOW SCHEMAS IN {CATALOG}"
    ).collect()
}

missing_schemas = (
    required_schemas
    - actual_schemas
)

assert not missing_schemas, (
    f"Missing schemas: {missing_schemas}"
)

print("Required schemas: PASS")

Required schemas: PASS


##Validate Gold tables

In [0]:
required_gold_tables = {
    "master_client",
    "master_client_attribute",
    "client_source_map",
    "survivorship_audit",
    "master_client_history",
}

actual_gold_tables = {
    row["tableName"]
    for row in spark.sql(
        f"SHOW TABLES IN {CATALOG}.gold"
    ).collect()
}

missing_gold = (
    required_gold_tables
    - actual_gold_tables
)

assert not missing_gold, (
    f"Missing Gold tables: {missing_gold}"
)

print("Gold cataloging: PASS")

Gold cataloging: PASS


##Validate PII tags

In [0]:
# ------------------------------------------------------------
# Read column-level classification metadata.
# ------------------------------------------------------------

tags = spark.sql(
    f"""
    SELECT
        table_name,
        column_name,
        tag_name,
        tag_value
    FROM {CATALOG}.information_schema.column_tags
    WHERE schema_name = 'gold'
      AND table_name = 'master_client'
    """
)

tags.show(truncate=False)

+-------------+-----------+-----------+-----------------+
|table_name   |column_name|tag_name   |tag_value        |
+-------------+-----------+-----------+-----------------+
|master_client|address    |pii        |address          |
|master_client|phone      |pii        |phone            |
|master_client|tax_id     |pii        |tax_id           |
|master_client|name       |pii        |direct_identifier|
|master_client|aum        |sensitivity|financial        |
|master_client|email      |pii        |email            |
+-------------+-----------+-----------+-----------------+



In [0]:
required_pii_columns = {
    "name",
    "phone",
    "email",
    "address",
    "tax_id",
}

tagged_pii_columns = {
    row["column_name"]
    for row in (
        tags
        .filter(
            F.col("tag_name") == "pii"
        )
        .collect()
    )
}

missing_pii_tags = (
    required_pii_columns
    - tagged_pii_columns
)

assert not missing_pii_tags, (
    f"Missing PII tags: {missing_pii_tags}"
)

print("PII classification tags: PASS")

PII classification tags: PASS


##Validate PII Masking

###Test A — Unauthorized user ( with PII masking)

In [0]:
%sql
SELECT
    master_client_id,
    name,
    phone,
    email,
    tax_id
FROM bfsi_mdm_dev.gold.master_client
LIMIT 10;

master_client_id,name,phone,email,tax_id
MC000003,JEMMA DEAN,******0356,l***@example.net,********003A
MC000007,MITCHELL JANICE BLACKBURN,******0059,d***@example.com,********007A
MC000009,Leanne Robert Stevens,******0874,r***@example.com,null
MC000012,Molly Webb,******0814,h***@example.com,null
MC000016,LEIGH BURTON,******8951,v***@example.net,********018A
MC000017,William Francesca Allan,******0479,b***@example.org,null
MC000018,GEOFFREY WEST,******0710,u***@example.net,********020A
MC000021,JOAN HUNT,******0116,l***@example.net,********023A
MC000023,LEONARD THOMAS SHAH,******0218,m***@example.com,********025A
MC000026,Joe Saunders,******8419,r***@example.com,null


###Test A — Authorized PII user

In [0]:
%sql
SELECT
    master_client_id,
    name,
    phone,
    email,
    tax_id
FROM bfsi_mdm_dev.gold.master_client
LIMIT 10;

master_client_id,name,phone,email,tax_id
MC000003,JEMMA DEAN,+44117 4960356,lewisbrian@example.net,QQ0000003A
MC000007,MITCHELL JANICE BLACKBURN,(0113)4960059,darrendouglas@example.com,QQ0000007A
MC000009,Leanne Robert Stevens,+44(0)9098790874,robertscarol@example.com,null
MC000012,Molly Webb,+44(0)116 496 0814,howard15@example.com,null
MC000016,LEIGH BURTON,029 2018951,vmurray@example.net,QQ0000018A
MC000017,William Francesca Allan,+44(0)118 496 0479,brookskatie@example.org,null
MC000018,GEOFFREY WEST,+44(0)2074960710,uwhite@example.net,QQ0000020A
MC000021,JOAN HUNT,(020) 7496 0116,liam07@example.net,QQ0000023A
MC000023,LEONARD THOMAS SHAH,0115 496 0218,mauricesmith@example.com,QQ0000025A
MC000026,Joe Saunders,(029)2018419,racheljones@example.com,null


## Unity Catalog Governance

### Catalog

bfsi_mdm_dev

### Governance hierarchy

Metastore
→ Catalog
→ Schema
→ Table / View / Function

## RBAC groups

- mdm_engineers
- mdm_stewards
- mdm_analysts
- mdm_pii_access

### PII classification

Golden Record PII columns:

- name
- phone
- email
- address
- tax_id

Sensitive financial attribute:

- aum

### Masking

Email, phone and tax_id are protected with Unity Catalog
column masks.

Members of mdm_pii_access can access original values.

Other authorized readers receive masked values.

### Lineage

Lineage was inspected through Unity Catalog Catalog Explorer and captured as screenshot evidence.

Direct queries to system.access.table_lineage and system.access.column_lineage have not been verified.

### Audit

bfsi_mdm_dev.audit.pipeline_run_audit records the project's application/pipeline publication audit.

Direct querying of Databricks platform audit data in system.access.audit has not been verified.

These are separate audit sources and must not be presented as interchangeable.

### Design boundary

Unity Catalog governs data access and observability.

It does not perform entity resolution, survivorship or
Golden Record construction.

Those operations remain in Databricks MDM processing.